# Program 5 – Autoencoder for Image Reconstruction

**Objective:** Train an autoencoder to reconstruct grayscale images.

**Dataset:** MNIST (default) or CIFAR-10 converted to grayscale (set `DATASET = "cifar10"`)

**Key tasks**
1. Design encoder and decoder using Dense layers.
2. Train to minimize reconstruction loss.
3. Visualize original vs reconstructed images.

## Theory (for viva)

An **autoencoder** is a neural network trained to output its own input: $\hat x \approx x$.

```
input (784) → [Encoder] → bottleneck / latent code (32) → [Decoder] → reconstruction (784)
```

- **Encoder** $z = f(x)$: compresses the image into a small latent vector.
- **Decoder** $\hat x = g(z)$: rebuilds the image from the latent vector.
- **Bottleneck:** smaller than the input, forcing the network to learn the most important features (not just copy pixels).
- **Unsupervised:** labels are not used — input is also the target.
- **Reconstruction loss:** Mean Squared Error $\frac{1}{n}\sum (x - \hat x)^2$ or Binary Cross-Entropy (pixels in [0,1]).
- **Uses:** dimensionality reduction, denoising, anomaly detection, pre-training.
- Output layer uses **sigmoid** so reconstructed pixels are in [0, 1].

## Step 1 – Import libraries

In [ ]:
import keras  # Keras 3 API (runs on TensorFlow backend)
from keras import layers  # Layer classes (Dense, Dropout, Embedding, ...)
import matplotlib.pyplot as plt  # Plotting graphs
import numpy as np  # Numerical arrays
import tensorflow as tf  # TensorFlow (tensors, GradientTape, seeds)

tf.random.set_seed(42)                  # Reproducibility

## Step 2 – Load data (MNIST or grayscale CIFAR-10)

In [ ]:
DATASET = "mnist"                                               # Change to "cifar10" to use grayscale CIFAR-10

if DATASET == "mnist":                                          # ----- MNIST branch -----
    (x_train, _), (x_test, _) = keras.datasets.mnist.load_data()   # Load images, ignore labels (unsupervised)
    IMG_H, IMG_W = 28, 28                                       # MNIST image size
    x_train = x_train.astype("float32") / 255.0                 # Scale pixels to [0, 1]
    x_test = x_test.astype("float32") / 255.0                   # Same for test
else:                                                           # ----- CIFAR-10 branch -----
    (x_train, _), (x_test, _) = keras.datasets.cifar10.load_data() # Load colour images (32x32x3)
    IMG_H, IMG_W = 32, 32                                       # CIFAR-10 image size
    x_train = tf.image.rgb_to_grayscale(x_train).numpy()        # Convert RGB -> 1 gray channel (32x32x1)
    x_test = tf.image.rgb_to_grayscale(x_test).numpy()          # Same for test
    x_train = x_train.astype("float32") / 255.0                 # Scale to [0, 1]
    x_test = x_test.astype("float32") / 255.0                   # Scale to [0, 1]

INPUT_DIM = IMG_H * IMG_W                                       # Number of pixels (784 for MNIST, 1024 for CIFAR)
x_train = x_train.reshape(-1, INPUT_DIM)                        # Flatten each image into a vector
x_test = x_test.reshape(-1, INPUT_DIM)                          # Flatten test images

print("Train:", x_train.shape, "Test:", x_test.shape)           # Check shapes

## Step 3 – Build encoder and decoder with Dense layers

In [ ]:
LATENT_DIM = 32                                                 # Size of the compressed code (bottleneck)

encoder = keras.Sequential([                                    # ----- Encoder -----
    keras.Input(shape=(INPUT_DIM,)),                            # Input: flattened image
    layers.Dense(256, activation="relu"),                       # Compress 784 -> 256
    layers.Dense(128, activation="relu"),                       # Compress 256 -> 128
    layers.Dense(LATENT_DIM, activation="relu"),                # Compress 128 -> 32 (latent code)
], name="encoder")                                              # Name shown in summary

decoder = keras.Sequential([                                    # ----- Decoder -----
    keras.Input(shape=(LATENT_DIM,)),                           # Input: 32-dim latent code
    layers.Dense(128, activation="relu"),                       # Expand 32 -> 128
    layers.Dense(256, activation="relu"),                       # Expand 128 -> 256
    layers.Dense(INPUT_DIM, activation="sigmoid"),              # Expand 256 -> 784, sigmoid keeps pixels in [0, 1]
], name="decoder")                                              # Name shown in summary

autoencoder = keras.Sequential([encoder, decoder], name="autoencoder")  # Full model = encoder followed by decoder

encoder.summary()                                               # Encoder architecture
decoder.summary()                                               # Decoder architecture
autoencoder.summary()                                           # Combined model

## Step 4 – Compile and train (input = target)

In [ ]:
autoencoder.compile(                                            # Configure training
    optimizer="adam",                                           # Adam optimizer
    loss="binary_crossentropy",                                 # Reconstruction loss (pixel-wise BCE)
    metrics=["mse"],                                            # Also report Mean Squared Error
)

history = autoencoder.fit(                                      # Train
    x_train, x_train,                                           # Input AND target are the same images
    epochs=20,                                                  # Number of epochs
    batch_size=256,                                             # Batch size
    shuffle=True,                                               # Shuffle each epoch
    validation_data=(x_test, x_test),                           # Validate reconstruction on test images
    verbose=2,                                                  # One line per epoch
)

## Step 5 – Plot reconstruction loss

In [ ]:
plt.figure(figsize=(8, 5))                                      # New figure
plt.plot(history.history["loss"], label="Train loss")           # Training reconstruction loss
plt.plot(history.history["val_loss"], label="Validation loss")  # Validation reconstruction loss
plt.xlabel("Epoch")                                             # X label
plt.ylabel("Binary cross-entropy")                              # Y label
plt.title("Autoencoder Reconstruction Loss")                    # Title
plt.legend()                                                    # Legend
plt.grid(True)                                                  # Grid
plt.show()                                                      # Show

## Step 6 – Visualize original vs reconstructed images

In [ ]:
codes = encoder.predict(x_test, verbose=0)                      # Encode test images -> (N, 32) latent vectors
reconstructed = decoder.predict(codes, verbose=0)               # Decode back -> (N, 784) images

n = 10                                                          # Number of images to show
plt.figure(figsize=(20, 4))                                     # Wide figure
for i in range(n):                                              # Loop over images
    ax = plt.subplot(2, n, i + 1)                               # Top row position
    plt.imshow(x_test[i].reshape(IMG_H, IMG_W), cmap="gray")    # Original image
    plt.title("Original")                                       # Label
    ax.axis("off")                                              # Hide axes

    ax = plt.subplot(2, n, i + 1 + n)                           # Bottom row position
    plt.imshow(reconstructed[i].reshape(IMG_H, IMG_W), cmap="gray")  # Reconstructed image
    plt.title("Reconstructed")                                  # Label
    ax.axis("off")                                              # Hide axes
plt.tight_layout()                                              # Tidy layout
plt.show()                                                      # Display

## Step 7 – Quantify reconstruction error

In [ ]:
mse_per_image = np.mean((x_test - reconstructed) ** 2, axis=1)  # MSE for each test image
print("Mean test MSE :", mse_per_image.mean())                  # Average reconstruction error
print("Compression   :", INPUT_DIM, "->", LATENT_DIM,
      f"({INPUT_DIM / LATENT_DIM:.1f}x smaller)")               # How much the code compresses the image

worst = np.argsort(mse_per_image)[-5:]                          # Indices of 5 worst-reconstructed images
plt.figure(figsize=(10, 4))                                     # Figure
for j, idx in enumerate(worst):                                 # Loop over worst images
    plt.subplot(2, 5, j + 1)                                    # Top row
    plt.imshow(x_test[idx].reshape(IMG_H, IMG_W), cmap="gray")  # Original
    plt.axis("off")                                             # Hide axes
    plt.subplot(2, 5, j + 6)                                    # Bottom row
    plt.imshow(reconstructed[idx].reshape(IMG_H, IMG_W), cmap="gray")  # Reconstruction
    plt.title(f"MSE={mse_per_image[idx]:.3f}")                  # Its error
    plt.axis("off")                                             # Hide axes
plt.suptitle("Hardest images to reconstruct")                   # Title
plt.tight_layout()                                              # Tidy
plt.show()                                                      # Show

## Expected observations
- Loss drops quickly in first few epochs, then slowly.
- Reconstructions look like slightly blurry versions of the originals — fine details are lost in the 32-dim bottleneck.
- Unusual handwriting has the highest error (this is the idea behind anomaly detection with autoencoders).
- Grayscale CIFAR-10 reconstructions are much blurrier (natural images are more complex than digits).

## Viva questions
1. Why is it called unsupervised? → No labels are used; the input is its own target.
2. What happens if the latent size equals the input size? → Network can learn the identity function and learns no useful features.
3. Why sigmoid in the final layer? → Pixel values are in [0, 1].
4. MSE vs BCE loss for autoencoders? → Both work for [0,1] pixels; BCE often gives sharper results on MNIST.
5. Autoencoder vs PCA? → Linear autoencoder with MSE ≈ PCA; non-linear activations let autoencoders learn more complex compressions.